# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. {"A": 2, "B": 1}. Потому что код по сути считает количество повторений букв.
2. Мой предикт - [0, 1, 1]. Функция является in-place, ничего не возвращает, а просто при вызове добавляет новый элемент 1 в список.
3. Формируется set (множество), но каждый элемент в множестве уникальный. Поэтому выйдет 2.
4. Я думаю выйдет {"ab": 2, "abc": 3}, потому что формируется хэш таблица (или словарь), а значения внутри ключей определяются длиной строки. Например, для "ab" у нас 2 так как 2 буквы, для "abc" - 3 соответственно.
5. Я предполагаю, что выйдет либо адрес функции, либо None. Функция не вызывается напрямую, потому что в коде ее инициализирует переменная x.

In [3]:
# проверка моих предсказаний

# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)

{'A': 2, 'B': 1}
[0, 1, 1]
2
{'ab': 2, 'abc': 3}
5
None


In [7]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [5]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [24]:
# База
#for i in range(1000, 1011):
#  print(rows[i]) просто тест для себя


def to_num(s):
  if type(s) == str:
    return float(s.replace(",", ".").replace(" ", ""))
  return float(s)


def clean_city(s):
  city = s.strip().title()
  names = {
        "Almaty": "Алматы",
        "Astana": "Астана",
        "Shymkent": "Шымкент"
    }
  return names.get(city, city) # дефолтное значение


cleaned_rows = []
for row in rows:
    new_row = row.copy()
    new_row["city"] = clean_city(row["city"])
    new_row["amount"] = to_num(row["amount"])
    cleaned_rows.append(new_row)

print("Города:", sorted(set(r["city"] for r in cleaned_rows)))

Города: ['Актобе', 'Алматы', 'Астана', 'Атырау', 'Караганда', 'Шымкент']


In [28]:
category_counts = {}

for row in cleaned_rows:
    category = row["category"]
    category_counts[category] = category_counts.get(category, 0) + 1

for category in sorted(category_counts, key=category_counts.get, reverse=True):
    print(category, category_counts[category])

Одежда 1529
Электроника 1185
Дом 1105
Красота 991
Книги 653


In [26]:
category_revenue = {}
category_paid_count = {}

for i in cleaned_rows:
    if i["status"] == "оплачен":
        category = row["category"]
        category_revenue[category] = category_revenue.get(category, 0) + row["amount"]
        category_paid_count[category] = category_paid_count.get(category, 0) + 1

for category, revenue in sorted(
    category_revenue.items(),
    key=lambda x: x[1],
    reverse=True
):
    avg = revenue / category_paid_count[category]
    print(f"{category}: выручка {revenue:.0f} ₸, средний чек {avg:.0f} ₸")

Электроника: выручка 65270240 ₸, средний чек 61287 ₸
Одежда: выручка 18321080 ₸, средний чек 13286 ₸
Дом: выручка 10516320 ₸, средний чек 10655 ₸
Красота: выручка 5507620 ₸, средний чек 6188 ₸
Книги: выручка 2294100 ₸, средний чек 3935 ₸


In [27]:
monthly_count = {}

for i in cleaned_rows:
    month = i["order_date"][:7]
    monthly_count[month] = monthly_count.get(month, 0) + 1

for month in sorted(monthly_count):
    print(f"{month}: {monthly_count[month]}")


2026-09: 2782
2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [29]:
# Стандарт
monthly_revenue = {}

for row in cleaned_rows:
    if row["status"] == "оплачен":
        month = row["order_date"][:7]
        monthly_revenue[month] = monthly_revenue.get(month, 0) + row["amount"]

for month in sorted(monthly_revenue):
    print(f"{month}: {monthly_revenue[month]:.0f} ₸")

september = monthly_revenue["2026-09"]
october = monthly_revenue["2026-10"]

change = (october - september) / september * 100
print(f"Изменение октября к сентябрю: {change:.1f}%")

2026-09: 57489850 ₸
2026-10: 44419510 ₸
Изменение октября к сентябрю: -22.7%


## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [ ]:
# Челлендж